# 💻 GeoAI Research Console Dashboard
## Module 08: Interactive Streamlit Dashboard

## 📌 Objective
Document the architecture, multi-page layout, interactive Streamlit component suite, backend service integration, output artifact loading, and temporal change detection workflow for the **GeoAI Research Console** web application.

---

## 🏗️ Streamlit Multi-Page Architecture & Folder Structure
The dashboard is built natively using Python and **Streamlit** (v1.30+), eliminating Flask dependencies, route handlers, and Jinja2 HTML templates.

```
dashboard/
├── app.py                     # Main Landing Page & Executive Summary Dashboard
└── pages/
    ├── 01_Dataset.py          # Interactive Dataset Explorer & Class Distributions
    ├── 02_Training.py         # Loss & Accuracy Epoch Progression Curves
    ├── 03_Evaluation.py       # Unified EuroSAT Test Evaluation & Confusion Matrix
    ├── 04_Comparison.py       # Tri-Model Comparative Benchmarking
    ├── 05_Change_Detection.py # Siamese Bi-Temporal Change Detection & File Uploaders
    ├── 06_Error_Analysis.py   # Top-5 Misclassified Sample Cards & Diagnostic Analysis
    ├── 07_UC_Merced.py        # External UC Merced Holdout Generalization Metrics
    └── 08_About.py            # Technical Methodology & Theoretical Background
```

---

## 🛠️ Interactive Streamlit Components & Widgets Used
1. `st.sidebar`: Global sidebar featuring the active dataset context selector (`EuroSAT` vs `UC Merced`) and metadata summary card.
2. `st.columns`: Responsive grid layout for metric cards, side-by-side plot comparisons, and top-5 misclassified sample cards.
3. `st.metric`: Displays high-impact KPI summary stats (Accuracy, Precision, Recall, F1, Cosine Similarity, Decision Threshold).
4. `st.file_uploader`: Native file upload widget for bi-temporal pre-change ($T_1$) and post-change ($T_2$) satellite/aerial image uploads.
5. `st.image`: Direct rendering of PIL Image objects, Matplotlib plots, difference maps, and heatmap overlays.
6. `st.dataframe`: Interactive tabular data view for model comparison metrics and error analysis sample logs.
7. `st.tabs`: Tabbed views for separating EuroSAT primary test evaluation and UC Merced holdout evaluation.
8. `st.success / st.warning / st.error`: Color-coded banners for change detection alerts and evaluation status.
9. `st.code`: Syntax-highlighted text block for classification reports.

---

## 🔄 Backend Service Integration & Workflow Pipelines

### 1. Change Detection Workflow
1. User uploads $T_1$ (Pre-change) and $T_2$ (Post-change) images via `st.file_uploader` in `05_Change_Detection.py`.
2. Images are processed through `services.change_detection_service.detect_change()`.
3. Fine-tuned ResNet18 backbone extracts 512-dimensional latent feature vectors $f(T_1), f(T_2) \in \mathbb{R}^{512}$.
4. Cosine similarity $\text{Sim}(f_1, f_2)$ and Cosine Distance $D = 1 - \text{Sim}$ are computed.
5. Distance $D$ is compared against the Youden J optimal decision threshold $\theta_{optimal} = 0.2698$ (ROC-AUC = 0.9966).
6. Confidence comparison bar chart, difference map, and spatial heatmap overlay are generated and displayed via `st.image`.

### 2. Evaluation Workflow
1. `services.metrics_service` reads evaluation metrics from `outputs/resnet18_finetuned/metrics.json` and `outputs/uc_merced/metrics.json`.
2. Classification reports are read from `classification_report.txt` and displayed using `st.code`.
3. Confusion matrix image artifacts (`confusion_matrix.png`) are rendered side-by-side.


### 1. Verify Streamlit Pages & App Structure

In [1]:
import sys
from pathlib import Path

# Smart Project Root Resolution
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'configs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

dashboard_dir = PROJECT_ROOT / 'dashboard'
app_file = dashboard_dir / 'app.py'
pages_dir = dashboard_dir / 'pages'

print('GeoAI Console Streamlit Architecture Verification:')
print(f'  Main Entry App: {app_file.name} (Exists: {app_file.exists()})')
print('  Registered Pages:')

if pages_dir.exists():
    for page in sorted(pages_dir.glob('*.py')):
        print(f'    - {page.name:25s} -> {page.relative_to(PROJECT_ROOT)}')

print('\n🚀 Launch Dashboard Commands:')
print('  1. python -m streamlit run dashboard/app.py')
print('  2. .venv/bin/streamlit run dashboard/app.py')

GeoAI Console Streamlit Architecture Verification:
  Main Entry App: app.py (Exists: True)
  Registered Pages:
    - 01_Dataset.py             -> dashboard/pages/01_Dataset.py
    - 02_Training.py            -> dashboard/pages/02_Training.py
    - 03_Evaluation.py          -> dashboard/pages/03_Evaluation.py
    - 04_Comparison.py          -> dashboard/pages/04_Comparison.py
    - 05_Change_Detection.py    -> dashboard/pages/05_Change_Detection.py
    - 06_Error_Analysis.py      -> dashboard/pages/06_Error_Analysis.py
    - 07_UC_Merced.py           -> dashboard/pages/07_UC_Merced.py
    - 08_About.py               -> dashboard/pages/08_About.py

🚀 Launch Dashboard Commands:
  1. python -m streamlit run dashboard/app.py
  2. .venv/bin/streamlit run dashboard/app.py


### 2. Backend Services & Output Artifact Integration

In [2]:
from dashboard.services import (
    metrics_service,
    change_detection_service,
    error_analysis_service,
    uc_merced_service,
)
from configs.dataset_info import get_dataset_info

all_metrics = metrics_service.get_all_metrics()

def fmt_acc(val):
    v = float(val) if val is not None else 0.0
    return f"{v * 100 if v <= 1.0 else v:.2f}%"

fine_val = all_metrics.get('resnet18_finetuned', {}).get('accuracy', 0.0)
base_val = all_metrics.get('baseline_cnn', {}).get('accuracy', 0.0)
froz_val = all_metrics.get('resnet18_frozen', {}).get('accuracy', 0.0)
ucm_val = all_metrics.get('uc_merced', {}).get('training_accuracy', all_metrics.get('uc_merced', {}).get('accuracy', 0.0))

print('✅ Dashboard Service Integration Verified:')
print('  EuroSAT Fine-Tuned Accuracy:', fmt_acc(fine_val))
print('  Baseline CNN Accuracy:', fmt_acc(base_val))
print('  Frozen ResNet18 Accuracy:', fmt_acc(froz_val))
print('  UC Merced Holdout Accuracy:', fmt_acc(ucm_val))

✅ Dashboard Service Integration Verified:
  EuroSAT Fine-Tuned Accuracy: 97.11%
  Baseline CNN Accuracy: 78.91%
  Frozen ResNet18 Accuracy: 89.80%
  UC Merced Holdout Accuracy: 96.19%


### 3. Verify Project Report Data Metadata JSON

In [3]:
import json

report_data_path = PROJECT_ROOT / 'outputs' / 'project_report_data.json'
if report_data_path.exists():
    with open(report_data_path, 'r', encoding='utf-8') as f:
        report_data = json.load(f)
    print('Project Report Data Metadata Loaded Successfully:')
    print('  Project Name:', report_data['project_information']['project_name'])
    print('  PyTorch Version:', report_data['project_information']['pytorch_version'])
    print('  Fine-Tuned Accuracy:', report_data['report_values']['test_accuracy'])
    print('  UC Merced Accuracy:', report_data['report_values']['uc_merced_accuracy'])
else:
    print('Warning: project_report_data.json not found at outputs/')

Project Report Data Metadata Loaded Successfully:
  Project Name: Deep Learning Land-Use Classification & Change Detection
  PyTorch Version: 2.13.0+cu130
  Fine-Tuned Accuracy: 97.11
  UC Merced Accuracy: 96.19


## 📝 Conclusion
The GeoAI Research Console dashboard has been completely migrated to **Streamlit**, providing a modern, interactive multi-page web application. The application can be launched using:

```bash
python -m streamlit run dashboard/app.py
```
or
```bash
.venv/bin/streamlit run dashboard/app.py
```